# System-GMM Robustness Checks

**Purpose.** Estimate dynamic panel models and report serial-correlation and over-identification diagnostics.

**Inputs.** Country-year transport, macroeconomic, and sectoral data.

**Outputs.** System-GMM summary tables.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Dynamic-panel robustness analysis dependencies

import pandas as pd
import numpy as np
from scipy.stats import norm
from linearmodels.iv import IVGMM
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("=" * 70)

In [ ]:
# Cell | Transport, macroeconomic and sectoral data ingestion

import glob
import os

region_mapping = {
    'North Africa': ['Algeria', 'Egypt', 'Libya', 'Mauritania', 'Morocco', 'Tunisia'],
    'West Africa': ['Benin', 'Burkina Faso', 'Ivory Coast', 'Gambia', 'Ghana',
                    'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger',
                    'Nigeria', 'Senegal', 'Sierra Leone', 'Togo'],
    'Central Africa': ['Cameroon', 'Central African Republic', 'Chad', 'Republic of the Congo',
                       'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon'],
    'East Africa': ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya',
                    'Rwanda', 'Somalia', 'South Sudan', 'Sudan', 'United Republic of Tanzania', 'Uganda'],
    'Southern Africa': ['Angola', 'Botswana', 'eSwatini', 'Lesotho', 'Madagascar', 'Malawi',
                        'Mozambique', 'Namibia', 'South Africa', 'Zambia', 'Zimbabwe']
}

region_map = {c: r for r, cs in region_mapping.items() for c in cs}
all_mapped_countries = list(region_map.keys())

transport_file = r'E:\Africa_Infrastructure_Data\transport_statistics_2015_2025.xlsx'
df_transport = pd.read_excel(transport_file)
df_transport.columns = df_transport.columns.str.strip()

road_type_cols = ['road_motorway_km', 'road_trunk_km', 'road_primary_km',
                  'road_secondary_km', 'road_tertiary_km']
df_transport['total_road_km'] = df_transport[road_type_cols].sum(axis=1)


econ_file = r'E:\Africa_Infrastructure_Data\afdb\ObservationData_fhkvkjf_constant.csv'
df_econ_raw = pd.read_csv(econ_file)

country_name_mapping = {
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Tanzania": "United Republic of Tanzania",
    "Eswatini": "eSwatini",
    "Cote d'Ivoire": "Ivory Coast"
}
df_econ_raw['country'] = df_econ_raw['country'].replace(country_name_mapping)
df_econ_raw['region'] = df_econ_raw['country'].map(region_map)
df_econ_raw = df_econ_raw[df_econ_raw['country'].isin(all_mapped_countries)].copy()

df_econ_raw['year'] = df_econ_raw['Date'].astype(int)
df_econ_raw['value'] = pd.to_numeric(df_econ_raw['Value'], errors='coerce')
df_econ_raw = df_econ_raw[(df_econ_raw['year'] >= 2015) & (df_econ_raw['year'] <= 2024)]

df_gdp = df_econ_raw[df_econ_raw['indicator'] == 'GDP (constant 2000 US$)'][
    ['country', 'year', 'value']
].rename(columns={'value': 'GDP'})

df_gdppc = df_econ_raw[df_econ_raw['indicator'] == 'GDP per capita,(constant 2000 Prices, US $)'][
    ['country', 'year', 'value']
].rename(columns={'value': 'GDP_PC'})

_pop_df  = df_econ_raw[df_econ_raw['indicator'] == 'Population, total'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'Pop'})
_hfce_df = df_econ_raw[df_econ_raw['indicator'] == 'Household final consumption expenditure (current US$)'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'HFCE'})
_er_df   = df_econ_raw[df_econ_raw['indicator'] == 'Exchange rate (LCU per US$, period average)'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'ER'})
_cpi_df  = df_econ_raw[df_econ_raw['indicator'] == 'Inflation, consumer prices  index (2000 = 100)'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'CPI'})

_mc = (_pop_df.merge(_hfce_df, on=['country', 'year'], how='inner')
              .merge(_er_df,   on=['country', 'year'], how='left')
              .merge(_cpi_df,  on=['country', 'year'], how='left'))
_mc['HFCE_pc_curr'] = _mc['HFCE'] / _mc['Pop']
_base = _mc[_mc['year'] == 2015][['country', 'ER', 'CPI']].rename(
    columns={'ER': 'ER_2015', 'CPI': 'CPI_2015'})
_mc = _mc.merge(_base, on='country', how='left')
_mc['ER_ratio']  = (_mc['ER']  / _mc['ER_2015'] ).fillna(1)
_mc['CPI_ratio'] = (_mc['CPI_2015'] / _mc['CPI']).fillna(1)
_mc['HFCE_PC'] = _mc['HFCE_pc_curr'] * _mc['ER_ratio'] * _mc['CPI_ratio']
df_cons2 = _mc[['country', 'year', 'HFCE_PC']]

df_macro = df_gdp.copy()
df_macro = df_macro.merge(df_gdppc, on=['country', 'year'], how='left')
df_macro = df_macro.merge(df_cons2, on=['country', 'year'], how='left')
df_macro['year'] = df_macro['year'].astype('Int64')

wb_dir = r'E:\Africa_Infrastructure_Data\world_bank'
sector_files = {
    'Agriculture_value_added': glob.glob(os.path.join(wb_dir, '**', 'API_NV.AGR*.csv'), recursive=True),
    'Industry_value_added':    glob.glob(os.path.join(wb_dir, '**', 'API_NV.IND*.csv'), recursive=True),
    'Services_value_added':    glob.glob(os.path.join(wb_dir, '**', 'API_NV.SRV*.csv'), recursive=True),
}

wb_to_transport_mapping = {
    'Congo, Dem. Rep.':  'Democratic Republic of the Congo',
    'Congo, Rep.':       'Republic of the Congo',
    'Tanzania':          'United Republic of Tanzania',
    'Eswatini':          'eSwatini',
    "Cote d'Ivoire":     'Ivory Coast',
    'Egypt, Arab Rep.':  'Egypt',
    'Gambia, The':       'Gambia',
}
YEAR_RANGE = list(range(2015, 2025))

def load_sector_long(files, sector_name):
    if not files:
        return pd.DataFrame()
    df = pd.read_csv(files[0], skiprows=4)
    df['Country Name'] = df['Country Name'].replace(wb_to_transport_mapping)
    df = df[df['Country Name'].isin(all_mapped_countries)].copy()
    year_cols = [str(y) for y in YEAR_RANGE if str(y) in df.columns]
    df = df[['Country Name', 'Country Code'] + year_cols]
    valid_countries = df.set_index('Country Name')[year_cols].notna().sum(axis=1)
    valid_countries = valid_countries[valid_countries > 3].index.tolist()
    df = df[df['Country Name'].isin(valid_countries)]
    df_long = df.melt(id_vars=['Country Name', 'Country Code'], value_vars=year_cols,
                      var_name='year', value_name=sector_name)
    df_long['year'] = df_long['year'].astype(int)
    return df_long

dfs = []
for sector, files in sector_files.items():
    df_sector = load_sector_long(files, sector)
    if not df_sector.empty:
        dfs.append(df_sector)

if dfs:
    df_industry = dfs[0]
    for df_s in dfs[1:]:
        df_industry = df_industry.merge(df_s, on=['Country Name', 'Country Code', 'year'], how='outer')
    df_industry = df_industry.rename(columns={'Country Name': 'country'})
    df_industry['year'] = pd.to_numeric(df_industry['year'], errors='coerce').astype('Int64')
    df_industry = df_industry[(df_industry['year'] >= 2015) & (df_industry['year'] <= 2024)]
else:
    df_industry = pd.DataFrame()


In [ ]:
# Cell | Raw macroeconomic indicator inspection
df_econ_raw['indicator'].unique()

In [ ]:
# Cell | Country-year macroeconomic and sectoral panels

_t_cols = ['total_road_km', 'motorway_trunk_paved_km', 'railway_length_km',
           'railway_station_area_km2', 'port_area_km2', 'airport_area_km2']
_macro_cols = ['GDP','GDP_PC','HFCE_PC']
_industry_cols = ['Agriculture_value_added', 'Industry_value_added', 'Services_value_added']

# Work on a copy to preserve the source table.
df_t = df_transport.copy()
df_t['year'] = pd.to_numeric(df_t['year'], errors='coerce').astype('Int64')
df_t = df_t[(df_t['year'] >= 2015) & (df_t['year'] <= 2024)]

# Merge records using the shared country-year keys.
df_panel_macro = df_t.merge(df_macro, on=['country', 'year'], how='left')
df_panel_macro['region'] = df_panel_macro['country'].map(region_map)
df_panel_macro = df_panel_macro.sort_values(['country', 'year']).reset_index(drop=True)


ind_cols = [c for c in df_industry.columns if c not in ['country', 'Country Code', 'year']]
df_panel_industry = df_t.merge(
    df_industry[['country', 'year'] + ind_cols], on=['country', 'year'], how='inner'
)
df_panel_industry['region'] = df_panel_industry['country'].map(region_map)
df_panel_industry = df_panel_industry.sort_values(['country', 'year']).reset_index(drop=True)



In [ ]:
# Cell | Log-level transformation of model variables

import numpy as np

# Apply log(1 + x) so zero-valued infrastructure measures remain defined.
def _compute_log_levels(df_src, value_cols, entity_col):
    """Compute log-transformed values."""
    df_res = df_src[[entity_col, 'year']].copy()
    for c in value_cols:
        vals = df_src[c].astype(float).clip(lower=0)
        df_res[f'{c}_log'] = np.log1p(vals)
    return df_res

_df_nat_log_m = _compute_log_levels(
    df_panel_macro, _t_cols + _macro_cols, entity_col='country')

_df_nat_log_i = _compute_log_levels(
    df_panel_industry, _t_cols + _industry_cols, entity_col='country')



In [ ]:
# Cell | System-GMM estimators

import numpy as np
import pandas as pd
from scipy.stats import norm
from linearmodels.iv import IVGMM


# Report Arellano-Bond AR(1) and AR(2) diagnostics from differenced residuals.
def _calc_ab_ar(df_resid, lag):
    """Run an Arellano-Bond serial-correlation test."""
    df_resid = df_resid.copy()
    # Create a within-country lagged variable.
    df_resid[f'resid_lag{lag}'] = df_resid.groupby(level=0)['resid'].shift(lag)
    valid = df_resid.dropna()
    if len(valid) < 10:
        return np.nan, np.nan
        
    e_t     = valid['resid'].values
    e_t_lag = valid[f'resid_lag{lag}'].values
    
    products = e_t * e_t_lag
    num = np.sum(products)
    
    den = np.std(products, ddof=1) * np.sqrt(len(products))
    if den == 0:
        return np.nan, np.nan
        
    stat  = num / den
    p_val = 2 * (1 - norm.cdf(abs(stat)))
    return stat, p_val


def run_system_gmm(df_panel, y_col, x_cols, entity_col='country', time_col='year', max_lag=2):
    """Estimate a system-GMM model."""
    if len(df_panel) < 30:
        return None

    df = df_panel[[entity_col, time_col, y_col] + x_cols].copy()
    # Order the panel by country and year before estimation.
    df = df.sort_values([entity_col, time_col]).set_index([entity_col, time_col])

    # Compute first differences within each country.
    # Difference the panel to remove time-invariant country effects.
    df_diff = df.groupby(level=0).diff()
    df['y_diff']      = df_diff[y_col]
    df['y_lag1']      = df.groupby(level=0)[y_col].shift(1)       # $y_{t-1}$
    df['y_lag1_diff'] = df.groupby(level=0)['y_diff'].shift(1)    # $\Delta y_{t-1}$
    for c in x_cols:
        df[f'{c}_diff'] = df_diff[c] # $\Delta x_t$

    # Use lagged levels as instruments for the differenced equation.
    iv_diff_cols  = []
    # Use lagged first differences as instruments for the level equation.
    iv_level_cols = []

    for lag in range(2, max_lag + 2):
        col_y = f'y_lag{lag}'
        df[col_y] = df.groupby(level=0)[y_col].shift(lag)
        iv_diff_cols.append(col_y)
        for x in x_cols:
            col_x = f'{x}_lag{lag}'
            df[col_x] = df.groupby(level=0)[x].shift(lag)
            iv_diff_cols.append(col_x)

    # Create a within-country lagged variable.
    df['y_diff_lag1'] = df.groupby(level=0)['y_diff'].shift(1)
    iv_level_cols.append('y_diff_lag1')
    for x in x_cols:
        col_xd = f'{x}_diff_lag1'
        df[col_xd] = df.groupby(level=0)[f'{x}_diff'].shift(1)
        iv_level_cols.append(col_xd)

    diff_vars  = ['y_diff', 'y_lag1_diff'] + [f'{c}_diff' for c in x_cols] + iv_diff_cols
    level_vars = [y_col, 'y_lag1'] + x_cols + iv_level_cols

    df_diff_clean  = df[df[diff_vars].notna().all(axis=1)].copy()
    df_level_clean = df[df[level_vars].notna().all(axis=1)].copy()
    if len(df_diff_clean) < 15 or len(df_level_clean) < 15:
        return None

    y_diff_c  = df_diff_clean['y_diff']
    X_diff_c  = df_diff_clean[['y_lag1_diff'] + [f'{c}_diff' for c in x_cols]]
    Z_diff_c  = df_diff_clean[iv_diff_cols]
    
    y_level_c = df_level_clean[y_col]
    X_level_c = df_level_clean[['y_lag1'] + x_cols]
    Z_level_c = df_level_clean[iv_level_cols]

    X_cols_std = [f'{y_col}_lag1'] + x_cols
    X_diff_c.columns  = X_cols_std
    X_level_c.columns = X_cols_std

    try:
        # Stack the equation-specific observations for joint estimation.
        y_stack = pd.concat([y_diff_c, y_level_c])
        X_stack = pd.concat([X_diff_c, X_level_c])
        n_diff  = len(y_diff_c)
        n_total = len(y_stack)

        Z_d = Z_diff_c.values
        Z_l = Z_level_c.values
        # Combine the difference- and level-equation instrument blocks for system GMM.
        Z_stack = np.zeros((n_total, Z_d.shape[1] + Z_l.shape[1]))
        Z_stack[:n_diff, :Z_d.shape[1]]  = Z_d  # Construct and screen the internal instrument matrix.
        Z_stack[n_diff:,  Z_d.shape[1]:] = Z_l  # Construct and screen the internal instrument matrix.

        Z_df = pd.DataFrame(Z_stack, index=y_stack.index)
        # Construct and screen the internal instrument matrix.
        Z_df = Z_df.loc[:, Z_df.var() > 1e-10]
        Z_df = Z_df.loc[:, ~Z_df.T.duplicated()]
        # Construct and screen the internal instrument matrix.
        if Z_df.shape[1] < X_stack.shape[1]:
            return None

        # Fit the GMM model with a robust weighting matrix.
        res = IVGMM(dependent=y_stack, exog=None, endog=X_stack,
                    instruments=Z_df, weight_type='robust').fit()

        # Calculate residual-based diagnostic quantities.
        resid_diff = res.resids.values[:n_diff]
        df_resid   = pd.DataFrame({'resid': resid_diff}, index=y_diff_c.index)
        ar1_stat, ar1_pval = _calc_ab_ar(df_resid, 1)
        ar2_stat, ar2_pval = _calc_ab_ar(df_resid, 2)
        
        # Store model-diagnostic statistics.
        hansen_stat = res.j_stat.stat if hasattr(res, 'j_stat') else np.nan
        hansen_pval = res.j_stat.pval if hasattr(res, 'j_stat') else np.nan

        return dict(method='Sys-GMM', coef=res.params.values, se=res.std_errors.values,
                    pval=res.pvalues.values, var_names=X_cols_std,
                    n_obs=len(y_stack), n_instruments=Z_df.shape[1],
                    ar1_stat=ar1_stat, ar1_pval=ar1_pval,
                    ar2_stat=ar2_stat, ar2_pval=ar2_pval,
                    hansen_stat=hansen_stat, hansen_pval=hansen_pval)
    except Exception as e:
        return None

In [ ]:
# Cell | Batch dynamic-panel estimation and diagnostics

print("\n" + "=" * 70)
print("=" * 70)

_e_cols    = ['GDP', 'GDP_PC', 'HFCE_PC', 'Agriculture_value_added', 'Industry_value_added', 'Services_value_added']
_t_cols    = ['total_road_km', 'motorway_trunk_paved_km', 'railway_length_km',
              'railway_station_area_km2', 'port_area_km2', 'airport_area_km2']
_macro_cols = ['GDP', 'GDP_PC', 'HFCE_PC']


print("-" * 70)

gmm_summary_log = []

for e_col in _e_cols:
    df_source_log = _df_nat_log_m.copy() if e_col in _macro_cols else _df_nat_log_i.copy()
    e_log_col = f'{e_col}_log'

    for t_col in _t_cols:
        t_log_col = f'{t_col}_log'
        if e_log_col not in df_source_log.columns or t_log_col not in df_source_log.columns:
            continue

        # Work on a copy to preserve the source table.
        # df_cur = df_source_log[['country', 'year', e_log_col, t_log_col]].copy()
        # df_cur[f'{t_log_col}_L1'] = df_cur.groupby('country')[t_log_col].shift(1)
        df_cur = df_source_log[['country', 'year', e_log_col, t_log_col]].copy()
        df_cur[f'{t_log_col}_L0'] = df_cur.groupby('country')[t_log_col].shift(0)

        df_cur = df_cur.drop(columns=[t_log_col]).dropna()
        if len(df_cur) < 30:
            continue

        # x_cols_cur = [f'{t_log_col}_L1']
        x_cols_cur = [f'{t_log_col}_L0']

        result = run_system_gmm(df_cur, e_log_col, x_cols_cur, max_lag=2)

        if result:
            gmm_summary_log.append({
                'Method':       result['method'],
                'Economic':     e_col,
                'Transport':    t_col,
                'rho':          result['coef'][0],
                'rho_pval':     result['pval'][0],
                'AR1_pval':     result['ar1_pval'],
                'AR2_pval':     result['ar2_pval'],
                'Hansen_pval':  result['hansen_pval'],
                'N':            result['n_obs'],
            })

if len(gmm_summary_log) > 0:
    df_gmm_log = pd.DataFrame(gmm_summary_log)


    print(df_gmm_log.to_string(index=False))

    ar1_pass    = (df_gmm_log['AR1_pval'] < 0.05).sum()
    ar2_pass    = (df_gmm_log['AR2_pval'] > 0.05).sum()
    hansen_pass = (df_gmm_log['Hansen_pval'] > 0.05).sum()
    rho_sig     = (df_gmm_log['rho_pval'] < 0.05).sum()
    n = len(df_gmm_log)

    for method in df_gmm_log['Method'].unique():
        df_m = df_gmm_log[df_gmm_log['Method'] == method]
        nm = len(df_m)
        a1 = (df_m['AR1_pval'] < 0.05).sum()
        a2 = (df_m['AR2_pval'] > 0.05).sum()
        hj = (df_m['Hansen_pval'] > 0.05).sum()
        rs = (df_m['rho_pval'] < 0.05).sum()

    for econ in df_gmm_log['Economic'].unique():
        df_e = df_gmm_log[df_gmm_log['Economic'] == econ]
        method_tag = df_e['Method'].iloc[0]
        a1 = (df_e['AR1_pval'] < 0.05).sum()
        a2 = (df_e['AR2_pval'] > 0.05).sum()
        hj = (df_e['Hansen_pval'] > 0.05).sum()
        rs = (df_e['rho_pval'] < 0.05).sum()

    _TRANS_LABEL = {
        'total_road_km':          'Road',
        'motorway_trunk_paved_km':'Paved Hwy',
        'railway_length_km':      'Railway',
        'railway_station_area_km2':'Rly Station',
        'port_area_km2':          'Port',
        'airport_area_km2':       'Airport'
    }
    for trans in df_gmm_log['Transport'].unique():
        df_t = df_gmm_log[df_gmm_log['Transport'] == trans]
        a1 = (df_t['AR1_pval'] < 0.05).sum()
        a2 = (df_t['AR2_pval'] > 0.05).sum()
        hj = (df_t['Hansen_pval'] > 0.05).sum()
        rs = (df_t['rho_pval'] < 0.05).sum()
else:
    pass

print("\n" + "=" * 70)
print("=" * 70)

In [ ]:
# Cell | GMM result-table inspection
df_gmm_log

In [ ]:
# Cell | Numerical summary of GMM estimates
numeric_df = df_gmm_log.select_dtypes(include="number")

summary = pd.DataFrame({
    "mean": numeric_df.mean(),
    "s.d.": numeric_df.std()
}).round(3)

summary